## 5 Python: Electric Field of a Line Charge

In this problem, work to extend the basis of numerical superposition to a line of charge.

Using numerical superposition, adding up the contributions to the electric field due to each chunk, solve the following problems.

1. Compute and represent the electric field of the charge at a distance of $0.01$m from the line charge along the $y$-axis

2. The analytical formula for the electric field of the rod at that location is $$ \frac{1}{4\pi\epsilon_0} \frac{Q}{\frac{L^2}{4} + y^2} $$ Compare the value of the electric field at that location for different values of "Nchunks", say for $10,20,50,100$, Identify how many chunks are needed to achieve less than a 1% relative error.

3. Using what you have build to find the electric field at this location, find the electric field at a variety of points round the line charge and represent them with arrows. Choose the locations, but be systematic.

### Part I

We briefly import our work from the given information.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
# Length of rod (m)
L = 0.2
# Location of the ends (m)
lineStart = -L/2
lineEnd = L/2

# Number of regions
nReg = 200

# Locations for regions, equally spaced
xLoc = np.linspace(lineStart+(L/2)/nReg,lineEnd-(L/2)/nReg, nReg)
yLoc = np.zeros(nReg)
# Stack locations, id est get (x,y) points
regLoc = np.stack((xLoc,yLoc),axis=1)


In [ ]:
fig = plt.figure(figsize = (10,6))
for i in regLoc:
    plt.plot(i[0],i[1],'ro')
plt.plot(0,0.01,'bo')
plt.show()

In [ ]:
k = 8.89e9 # electric constant, SI units
Q = 1e-7 # Charge in C
regCharge = Q / nReg

Now, we begin.

We can immediately see that our `regLoc` variable corresponds to $\bm{r'}$.

In [ ]:
rVec = [0,0.01] # in m

So `rVec` corresponds to $\bm{r}$.

In [ ]:
def FindScriptR(r,rPrime): #we create a function that takes the vectors and outputs scriptr.
    """This is for cartesian co-ordinates only.
        Accepts two vector inputs."""
    if len(r) != len(rPrime):
        raise IndexError("r and rPrime vectors are not equivalent in dimension!")
    scriptR = []
    for i in range(len(r)):
        scriptR.append(r[i]-rPrime[i])
    return np.array(scriptR)

def ElectricFieldPoint(r,rPrime,regCharge): #create a function that evaluates at a point the electric field.
    bScriptR = FindScriptR(r,rPrime)
    scriptR = np.linalg.norm(bScriptR)
    eVec = []
    for i in range(len(bScriptR)):
        e_i = k * regCharge * bScriptR[i] * (1/(scriptR**3))
        eVec.append(e_i)
    #print(scriptR) #for debugging purposes
    return np.array(eVec)

def SummationIntegration(r,rPrimeList,regCharge): #create a function that sums over all source points for a given $r$.
    I = np.zeros(len(r))
    #print(I) for debugging
    for rPrime in rPrimeList:
        I+= ElectricFieldPoint(r,rPrime,regCharge)
    return I

In [ ]:
print(SummationIntegration(rVec,regLoc,regCharge))

We can approximate $3.6266\cdot10^{-11}$ as $0$, and now we have $E = 8.845881\cdot10^{5}\hat{\bm{y}} N/C$.

### Part II

We check analytic formula directly:

In [ ]:
def AnalyticELine(y):
    res = k*Q*(1/(y*np.sqrt((L**2)/4+y**2)))
    return res

In [ ]:
valCalc = AnalyticELine(0.01)
print(f"{valCalc:.6e}")

Now, we try different values of `nReg`.

In [ ]:
def RegionError(nReg,r):
    # Locations for regions, equally spaced
    xLoc = np.linspace(lineStart+(L/2)/nReg,lineEnd-(L/2)/nReg, nReg)
    yLoc = np.zeros(nReg)
    # Stack locations, id est get (x,y) points
    regLoc = np.stack((xLoc,yLoc),axis=1)
    regCharge = Q / nReg
    E = SummationIntegration(r,regLoc,regCharge)
    valNum = E[1]
    relError = (np.abs(valCalc-valNum)/np.abs(valCalc)) * 100
    return float(relError)

In [ ]:
nRegs = [10,20,50,100]
rVec = [0,0.01]
for i in nRegs:
    relError = RegionError(i,rVec)
    print(f"Relative error for N={i} is {relError:.2e}%.")

From this, it should be evident that at around $20<N<25$, the error should get within 1%.

### Part III

We create a list of points.

First, we want to analyze the behavior at the ends of the line, so we choose 10 points around each end, 2 of them on the $x$-axis, the other 8 as a 2x2 grid above and below.
Then, we want to see the 6 points above and below the x axis, such that there are two rows above and below. The collation of these points are,

In [ ]:
points=np.array([
    (-0.15,-0.02),(-0.15,-0.01),(-0.15,0),(-0.15,0.01),(-0.15,0.02),
    (-0.05,-0.02),(-0.05,-0.015),(-0.05,0.015),(-0.05,0.02),
    (0,-0.02),(0,-0.015),(0,0.015),(0,0.02),
    (0.05,-0.02),(0.05,-0.015),(0.05,0.015),(0.05,0.02),
    (0.15,-0.02),(0.15,-0.01),(0.15,0),(0.15,0.01),(0.15,0.02)
])

In [ ]:
X = points[:,0]
Y = points[:,1]

In [ ]:
E_vecs = []
for i in points:
    E_vecs.append(SummationIntegration(i,regLoc,regCharge))
E_field=np.array(E_vecs)

E_x = E_field[:,0]
E_y = E_field[:,1]

In [ ]:
plt.figure(figsize=(10,6))
for i in regLoc:
    plt.plot(i[0],i[1],'ro')
plt.quiver(X,Y,E_x,E_y,angles='xy',color='b')

plt.xlim((-0.2,0.2))
plt.ylim((-0.03,0.03))
plt.show()